In [1]:
import sys, torch
sys.path.insert(0, "qwen3")                      # match the architecture of the checkpoint
from model import TinyQwen
from tokenizer import CharTokenizer


/Users/omrkr/Desktop/single_letter_transformers/.venv/lib/python3.13/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [ ]:
ckpt = torch.load("deepseek/tiny_qwen.pt", map_location="cpu", weights_only=False)
tok = CharTokenizer(ckpt["chars"])
model = TinyQwen(ckpt["cfg"]); model.load_state_dict(ckpt["model"]); model.eval()

model

TinyQwen(
  (embed_tokens): Embedding(30, 32)
  (layers): ModuleList(
    (0-1): 2 x TransformerBlock(
      (input_layernorm): RMSNorm()
      (attn): Attention(
        (q_proj): Linear(in_features=32, out_features=32, bias=False)
        (k_proj): Linear(in_features=32, out_features=16, bias=False)
        (v_proj): Linear(in_features=32, out_features=16, bias=False)
        (o_proj): Linear(in_features=32, out_features=32, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (post_attention_layernorm): RMSNorm()
      (mlp): MLP(
        (gate_proj): Linear(in_features=32, out_features=64, bias=False)
        (up_proj): Linear(in_features=32, out_features=64, bias=False)
        (down_proj): Linear(in_features=64, out_features=32, bias=False)
      )
    )
  )
  (norm): RMSNorm()
  (lm_head): Linear(in_features=32, out_features=30, bias=False)
)

In [3]:
model.embed_tokens.weight

Parameter containing:
tensor([[-2.5944e-02, -1.6900e-01, -3.9246e-01, -6.5868e-01,  6.8594e-01,
          2.8532e-02,  8.0508e-01,  1.3097e-02,  2.2318e-01,  1.0939e+00,
         -1.2547e+00, -4.1369e-01,  2.7395e-02, -3.2181e-01, -9.3094e-01,
          1.4433e+00,  1.2704e+00, -2.8189e-01,  3.3240e-01,  7.3917e-01,
         -1.7423e+00,  4.6255e-01,  1.3125e+00,  5.9483e-01,  2.2652e-01,
         -1.3165e+00, -1.1114e+00, -3.3506e-01,  4.1677e-01, -5.5589e-01,
          1.4281e+00,  2.4521e+00],
        [-6.3702e-01, -2.9985e-01,  7.1826e-01,  2.2338e-01,  1.0081e-02,
          8.4068e-01, -1.0365e+00, -3.6484e-01, -5.1324e-01, -8.9334e-01,
          5.1905e-01, -1.3854e+00, -1.0952e+00,  4.6777e-01, -4.6868e-01,
         -6.0839e-01,  1.5581e+00, -6.8748e-01,  1.2181e+00, -2.0418e-01,
         -1.2345e+00,  1.9673e+00,  2.5926e+00, -1.5755e+00,  1.1150e+00,
         -1.0973e+00,  5.7085e-01, -2.4900e-01,  4.6836e-01,  1.5202e+00,
          1.4613e+00, -3.1769e-01],
        [-5.0053e-

In [7]:
tok.encode("a")

[1]

In [6]:
model.embed_tokens.weight[1]

tensor([-0.6370, -0.2998,  0.7183,  0.2234,  0.0101,  0.8407, -1.0365, -0.3648,
        -0.5132, -0.8933,  0.5190, -1.3854, -1.0952,  0.4678, -0.4687, -0.6084,
         1.5581, -0.6875,  1.2181, -0.2042, -1.2345,  1.9673,  2.5926, -1.5755,
         1.1150, -1.0973,  0.5709, -0.2490,  0.4684,  1.5202,  1.4613, -0.3177],
       grad_fn=<SelectBackward0>)

In [4]:
start = torch.full((10, 1), tok.eos_id, dtype=torch.long)   # every name starts at EOS/newline
out = model.generate(start, max_new_tokens=model.cfg.max_seq_len,
                     temperature=0.8, eos_id=tok.eos_id)     # stops each row at EOS
out

tensor([[ 0,  2, 26,  7, 29, 17,  1,  0,  0],
        [ 0, 24,  5, 19,  9, 14,  0,  0,  0],
        [ 0,  4,  5, 21,  1, 12,  0,  0,  0],
        [ 0, 25, 23,  8,  1, 14,  0,  0,  0],
        [ 0,  9, 17,  7, 26, 12,  0,  0,  0],
        [ 0, 13,  9,  5, 12,  9,  8,  0,  0],
        [ 0,  8,  1,  0,  0,  0,  0,  0,  0],
        [ 0, 25, 23,  7, 26, 12,  0,  0,  0],
        [ 0, 18,  5,  4,  1, 19,  0,  0,  0],
        [ 0,  1, 12, 16,  5, 17,  5, 14,  0]])

In [5]:
for row in out.tolist():
    print(tok.decode(row[1:]).split("\n")[0])

bügşra
çetin
deval
özhan
irgül
mielih
ha
özgül
sedat
alperen
